# Text Classification with DistilBERT — Domain Adaptation for Nigerian Text

This notebook explores using HuggingFace transformer models (primarily DistilBERT) for text classification tasks involving Nigerian English — including domain-specific vocabulary, pidgin fragments, and code-switched text.

## Use Cases Explored

- Classifying text segments by domain (religious, technical, conversational)
- Detecting code-switching in transcribed speech
- Matching transcribed text fragments to reference text (e.g. verse matching)

## Why DistilBERT?

DistilBERT is a distilled (compressed) version of BERT — 40% smaller, 60% faster, retaining ~97% of BERT's performance on most NLP benchmarks. For low-resource or deployment-constrained environments (e.g. running locally on modest hardware), it's a practical starting point before scaling to larger models.


## Setup

In [ ]:
# !pip install transformers torch datasets

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import numpy as np

## 1. Zero-Shot Classification Pipeline

Before any fine-tuning, we can use a zero-shot classification pipeline to get a baseline sense of how well off-the-shelf models handle Nigerian text.

In [ ]:
# Zero-shot classifier — no training needed
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

# Example Nigerian English text samples
samples = [
    "See ehn, the thing wey you do no make sense at all",           # Pidgin
    "John 3:16 — For God so loved the world that he gave his only begotten son",  # Religious/biblical
    "The pressure drop across the valve was recorded at 2.3 bar",   # Technical
    "E don happen before, e go happen again — that na life",         # Pidgin proverb
    "We thank God for this opportunity to gather here today",        # Religious, Nigerian formal English
]

candidate_labels = ['religious', 'technical', 'conversational', 'pidgin']

for text in samples:
    result = classifier(text, candidate_labels)
    top = result['labels'][0]
    score = result['scores'][0]
    print(f'Text: "{text[:60]}..."')
    print(f'  → Predicted: {top} ({score:.2%})')
    print()

## 2. DistilBERT Feature Extraction

Extract sentence embeddings using DistilBERT. These can be used for similarity matching — e.g. finding the closest Bible verse to a transcribed fragment.

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
model.eval()

def get_embedding(text):
    """
    Get a sentence-level embedding from DistilBERT using mean pooling over token embeddings.
    Useful for semantic similarity tasks.
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        outputs = model.distilbert(**inputs)
    # Mean pool over token dimension
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding


def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# Example: match a transcribed fragment to candidate Bible verses
transcribed = "for God so loved the world"

candidates = [
    "For God so loved the world that he gave his only begotten Son",
    "In the beginning was the Word and the Word was with God",
    "The Lord is my shepherd I shall not want",
    "I can do all things through Christ who strengthens me",
]

query_emb = get_embedding(transcribed)
print(f'Query: "{transcribed}"\n')
scores = []
for c in candidates:
    sim = cosine_similarity(query_emb, get_embedding(c))
    scores.append((sim, c))

scores.sort(reverse=True)
print('Ranked matches:')
for score, verse in scores:
    print(f'  {score:.4f} — "{verse[:70]}"')

## 3. Observations on Nigerian English & Pidgin

### What works out of the box
- Standard Nigerian English (formal register) is handled well by DistilBERT — tokenization is clean, embeddings are semantically meaningful
- Religious text in English (even Nigerian church-style formal English) clusters correctly in embedding space
- Semantic similarity matching works reliably for English-only text

### Where it breaks down
- **Nigerian Pidgin**: Highly OOV (out of vocabulary) for most English-trained tokenizers. Words like "ehn", "wahala", "abi", "sha" are split into subword tokens in unexpected ways, degrading embedding quality
- **Code-switching**: A sentence that mixes Yoruba and English mid-utterance gets tokenized inconsistently — Yoruba words may be treated as misspelled English
- **Tonal diacritics**: Yoruba orthography uses diacritics (e.g. ẹ, ọ, à, á) which are either stripped or handled poorly by standard English tokenizers

### Implication
For robust Nigerian NLP — especially Yoruba or Pidgin — a tokenizer and model trained on or adapted for Nigerian language data is needed. Standard DistilBERT is a useful baseline but not a production solution for code-switched or low-resource Nigerian language tasks.

## 4. Fine-Tuning Setup (Outline)

To fine-tune DistilBERT for a Nigerian-specific classification task, the setup would look like this:

In [ ]:
from transformers import Trainer, TrainingArguments
from datasets import Dataset

# Example dataset structure
# Each example: {'text': str, 'label': int}
# Labels: 0=english, 1=pidgin, 2=yoruba, 3=code-switched

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    logging_dir='./logs',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Trainer would be initialized with:
# Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_eval,
# )

print('Fine-tuning setup ready — pending labelled Nigerian language dataset.')
print('Data collection and annotation is the current bottleneck.')

## Next Steps

- Collect labelled Nigerian text data across English, Pidgin, Yoruba, and code-switched categories
- Explore multilingual models (e.g. `xlm-roberta-base`) which may handle Yoruba better out of the box
- Investigate existing Yoruba NLP resources (JW300, Menyo-20k, OPUS corpora)
- Fine-tune on a language identification task as a first step before domain classification